In [2]:
import os
import json
import re
import random
import time
from openai import OpenAI

# ===================== 配置 =====================
api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("请设置 DEEPSEEK_API_KEY 环境变量")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
)

# ===================== 系统提示词 =====================
SYSTEM_PROMPT = """
你是一个资深的小红书爆款文案专家，擅长结合最新潮流和产品卖点，创作引人入胜、高互动、高转化的笔记文案。
你的任务是根据用户提供的产品和需求，生成包含标题、正文、相关标签和表情符号的完整小红书笔记。
请始终采用'Thought-Action-Observation'模式进行推理和行动。
文案风格需活泼、真诚、富有感染力。
完成任务后，请以JSON格式直接输出最终文案，格式如下：
{
  "title": "小红书标题",
  "body": "小红书正文",
  "hashtags": ["标签1","标签2","标签3","标签4","标签5"],
  "emojis": ["✨","🔥","💖"]
}
"""

# ===================== 工具定义 =====================
TOOLS_DEFINITION = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "搜索互联网上的实时信息，用于获取最新新闻、流行趋势、用户评价、行业报告等。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "搜索关键词"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "query_product_database",
            "description": "查询内部产品数据库，获取指定产品的详细卖点、成分、适用人群等。",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "产品名称"
                    }
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_emoji",
            "description": "根据文本内容生成小红书风格表情符号。",
            "parameters": {
                "type": "object",
                "properties": {
                    "context": {
                        "type": "string",
                        "description": "文案关键内容"
                    }
                },
                "required": ["context"]
            }
        }
    }
]

# ===================== 模拟工具函数 =====================
def mock_search_web(query: str) -> str:
    print(f"[Tool Call] 搜索网页：{query}")
    time.sleep(0.5)
    if "美妆趋势" in query:
        return "流行：早C晚A、伪素颜、屏障修复、氛围感、抗老"
    elif "面膜" in query:
        return "用户痛点：干皮、卡粉、泛红、熬夜暗沉"
    else:
        return f"关于 {query} 的信息：补水、保湿、修护是用户最关注的点"

def mock_query_product_database(product_name: str) -> str:
    print(f"[Tool Call] 查询产品：{product_name}")
    time.sleep(0.5)
    if "深海蓝藻保湿面膜" in product_name:
        return "深海蓝藻提取物，深层补水、修护屏障、敏感肌可用，吸收快不粘腻"
    return "产品信息：补水保湿、提亮肤色、温和不刺激"

def mock_generate_emoji(context: str) -> list:
    print(f"[Tool Call] 生成表情，上下文：{context}")
    if "补水" in context:
        return ["💦", "💧", "✨", "🌊"]
    return ["✨", "🔥", "💖", "💯", "🎉"]

available_tools = {
    "search_web": mock_search_web,
    "query_product_database": mock_query_product_database,
    "generate_emoji": mock_generate_emoji,
}

# ===================== 核心文案生成函数 =====================
def generate_rednote(product_name: str, tone_style: str = "活泼甜美", max_iterations: int = 5):
    print(f"\n🚀 生成小红书文案：{product_name}，风格：{tone_style}\n")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"请为产品「{product_name}」生成一篇小红书爆款文案。语气{tone_style}，包含标题、正文、标签、表情。输出严格JSON格式。"}
    ]

    iteration = 0
    while iteration < max_iterations:
        iteration += 1
        print(f"\n-- 第 {iteration} 轮迭代 --")

        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=messages,
                tools=TOOLS_DEFINITION,
                tool_choice="auto"
            )

            msg = response.choices[0].message

            # 如果调用工具
            if msg.tool_calls:
                print("✅ 模型决定调用工具")
                messages.append(msg)

                for tool_call in msg.tool_calls:
                    func_name = tool_call.function.name
                    args = json.loads(tool_call.function.arguments or "{}")

                    print(f"🔧 调用工具：{func_name} 参数：{args}")
                    func = available_tools[func_name]
                    result = func(**args)
                    print(f"📄 工具返回：{result}")

                    messages.append({
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "content": str(result)
                    })

            # 如果返回最终内容
            elif msg.content:
                print(f"\n📝 模型返回：{msg.content[:100]}...")

                # 提取 JSON
                match = re.search(r"```json\s*(.*?)\s*```", msg.content, re.DOTALL)
                json_str = match.group(1) if match else msg.content

                try:
                    data = json.loads(json_str)
                    print("\n🎉 文案生成成功！")
                    return json.dumps(data, ensure_ascii=False, indent=2)
                except:
                    messages.append(msg)
                    continue

        except Exception as e:
            print(f"❌ 错误：{e}")
            break

    return "❌ 生成失败"

# ===================== 测试运行 =====================
if __name__ == "__main__":
    result = generate_rednote("深海蓝藻保湿面膜", "活泼甜美")
    print("\n===== 最终文案 =====")
    print(result)


🚀 生成小红书文案：深海蓝藻保湿面膜，风格：活泼甜美


-- 第 1 轮迭代 --
✅ 模型决定调用工具
🔧 调用工具：search_web 参数：{'query': '深海蓝藻保湿面膜 2025 小红书 爆款 评价'}
[Tool Call] 搜索网页：深海蓝藻保湿面膜 2025 小红书 爆款 评价
📄 工具返回：用户痛点：干皮、卡粉、泛红、熬夜暗沉
🔧 调用工具：query_product_database 参数：{'product_name': '深海蓝藻保湿面膜'}
[Tool Call] 查询产品：深海蓝藻保湿面膜
📄 工具返回：深海蓝藻提取物，深层补水、修护屏障、敏感肌可用，吸收快不粘腻

-- 第 2 轮迭代 --
✅ 模型决定调用工具
🔧 调用工具：search_web 参数：{'query': '2025年热门面膜 护肤趋势 深层补水修护'}
[Tool Call] 搜索网页：2025年热门面膜 护肤趋势 深层补水修护
📄 工具返回：用户痛点：干皮、卡粉、泛红、熬夜暗沉
🔧 调用工具：generate_emoji 参数：{'context': '深海蓝藻面膜 补水 水润 修护 敏感肌 嫩滑 急救面膜'}
[Tool Call] 生成表情，上下文：深海蓝藻面膜 补水 水润 修护 敏感肌 嫩滑 急救面膜
📄 工具返回：['💦', '💧', '✨', '🌊']

-- 第 3 轮迭代 --

📝 模型返回：好的，信息收集完毕！现在来为你精心创作这篇爆款文案。

```json
{
  "title": "💦一敷回春！沙漠干皮救星🌊深海蓝藻面膜我直接封神！",
  "body": "姐妹们！！看我发现了什...

🎉 文案生成成功！

===== 最终文案 =====
{
  "title": "💦一敷回春！沙漠干皮救星🌊深海蓝藻面膜我直接封神！",
  "body": "姐妹们！！看我发现了什么宝藏面膜！！😭😭\n\n本沙漠大干皮一到换季就疯狂起皮卡粉，每次化妆都像灾难现场…直到被闺蜜安利了这瓶「深海蓝藻保湿面膜」！！用过一次就彻底沦陷了！💘\n\n🔥【先夸质地】\n它是那种QQ弹弹的果冻凝胶质地🍮，上脸冰冰凉凉的，夏天用真的爽到飞起！完全不粘腻，吸收巨快～敷15分钟揭下来，脸蛋像喝饱了水一样嘭起来！💧\n\n🌊【核心成分】\n深海蓝藻提取物真的不是